# Ligand-Receptor Interaction Analysis with Squidpy

Companion notebook for: [spatiabio.com — post 9](https://www.spatiabio.com/2026/07/ligand-receptor-interactions-squidpy.html)

Test ligand-receptor interactions between spatially adjacent Leiden clusters using `sq.gr.ligrec` (CellPhoneDB permutation framework).

In [ ]:
import squidpy as sq
import scanpy as sc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
adata = sq.datasets.visium_hne_adata()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='cell_ranger')
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.leiden(adata, key_added='cluster')
sq.gr.spatial_neighbors(adata)

In [ ]:
sq.gr.ligrec(
    adata,
    cluster_key='cluster',
    n_perms=1000,
)

In [ ]:
sq.pl.ligrec(
    adata,
    cluster_key='cluster',
    source_groups=['0', '1', '2'],
    target_groups=['0', '1', '2'],
    means_range=(0.3, float('inf')),
    alpha=0.001,
    swap_axes=True,
)

In [ ]:
# Extract top significant interactions between cluster 0 and cluster 1
means = adata.uns['cluster_ligrec']['means']
pvals = adata.uns['cluster_ligrec']['pvalues']

sig = means.loc[:, ('0', '1')].to_frame('mean')
sig['pval'] = pvals.loc[:, ('0', '1')]
sig = sig[(sig['mean'] > 0.3) & (sig['pval'] < 0.05)].sort_values('mean', ascending=False)
print(sig.head(10))